In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/spaceship-titanic/sample_submission.csv
/kaggle/input/competitions/spaceship-titanic/train.csv
/kaggle/input/competitions/spaceship-titanic/test.csv


In [2]:
# =========================================
# SPACESHIP TITANIC - HIGH SCORE SOLUTION
# =========================================

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import warnings
warnings.filterwarnings('ignore')

# =========================================
# LOAD DATA
# =========================================

train = pd.read_csv('/kaggle/input/competitions/spaceship-titanic/train.csv')
test = pd.read_csv('/kaggle/input/competitions/spaceship-titanic/test.csv')

train_len = len(train)

df = pd.concat([train, test], axis=0).reset_index(drop=True)

# =========================================
# FEATURE ENGINEERING
# =========================================

# -------- Cabin Split --------

df['Deck'] = df['Cabin'].fillna('Unknown/0/Unknown').apply(lambda x: x.split('/')[0])

df['CabinNum'] = df['Cabin'].fillna('Unknown/0/Unknown').apply(
    lambda x: x.split('/')[1]
)

df['Side'] = df['Cabin'].fillna('Unknown/0/Unknown').apply(
    lambda x: x.split('/')[2]
)

df['CabinNum'] = pd.to_numeric(df['CabinNum'], errors='coerce')

# -------- Group Size --------

df['Group'] = df['PassengerId'].apply(lambda x: x.split('_')[0])

group_counts = df['Group'].value_counts()

df['GroupSize'] = df['Group'].map(group_counts)

# -------- Spending Features --------

spend_cols = [
    'RoomService',
    'FoodCourt',
    'ShoppingMall',
    'Spa',
    'VRDeck'
]

for col in spend_cols:
    df[col] = df[col].fillna(0)

df['TotalSpend'] = df[spend_cols].sum(axis=1)

df['IsZeroSpend'] = (df['TotalSpend'] == 0).astype(int)

# -------- Age Features --------

df['Age'] = df['Age'].fillna(df['Age'].median())

df['AgeGroup'] = pd.cut(
    df['Age'],
    bins=[0, 12, 18, 30, 50, 100],
    labels=['Child', 'Teen', 'Young', 'Adult', 'Senior']
)

# -------- CryoSleep --------

df['CryoSleep'] = df['CryoSleep'].fillna(False)

# If CryoSleep = True, spending likely = 0
df.loc[df['CryoSleep'] == True, spend_cols] = \
    df.loc[df['CryoSleep'] == True, spend_cols].fillna(0)

# -------- HomePlanet --------

df['HomePlanet'] = df['HomePlanet'].fillna(
    df.groupby('Deck')['HomePlanet'].transform(lambda x: x.mode()[0] if not x.mode().empty else 'Earth')
)

# -------- Destination --------

df['Destination'] = df['Destination'].fillna(
    df['Destination'].mode()[0]
)

# -------- VIP --------

df['VIP'] = df['VIP'].fillna(False)

# =========================================
# DROP UNUSED
# =========================================

drop_cols = [
    'Cabin',
    'Name',
    'PassengerId',
    'Group'
]

df.drop(columns=drop_cols, inplace=True)

# =========================================
# LABEL ENCODING
# =========================================

cat_cols = df.select_dtypes(include='object').columns.tolist()

cat_cols += ['AgeGroup']

for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

# Boolean -> int

bool_cols = df.select_dtypes(include='bool').columns

for col in bool_cols:
    df[col] = df[col].astype(int)

# =========================================
# SPLIT BACK
# =========================================

train_df = df[:train_len]
test_df = df[train_len:]

X = train_df.drop('Transported', axis=1)
y = train_df['Transported'].astype(int)

X_test = test_df.drop('Transported', axis=1)

# =========================================
# CROSS VALIDATION
# =========================================

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# =========================================
# MODELS
# =========================================

lr_model = LogisticRegression(
    max_iter=500
)

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_split=5,
    random_state=42
)

xgb_model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    random_state=42
)

lgb_model = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

cat_model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.03,
    depth=6,
    loss_function='Logloss',
    verbose=0,
    random_state=42
)

models = {
    'LogisticRegression': lr_model,
    'RandomForest': rf_model,
    'XGBoost': xgb_model,
    'LightGBM': lgb_model,
    'CatBoost': cat_model
}

# =========================================
# TRAIN + VALIDATE
# =========================================

scores = {}

for name, model in models.items():

    fold_scores = []

    for train_idx, valid_idx in skf.split(X, y):

        X_train = X.iloc[train_idx]
        X_valid = X.iloc[valid_idx]

        y_train = y.iloc[train_idx]
        y_valid = y.iloc[valid_idx]

        model.fit(X_train, y_train)

        preds = model.predict(X_valid)

        score = accuracy_score(y_valid, preds)

        fold_scores.append(score)

    scores[name] = np.mean(fold_scores)

# =========================================
# PRINT SCORES
# =========================================

print("\n===== CROSS VALIDATION SCORES =====")

for k, v in scores.items():
    print(f"{k}: {v:.5f}")

# =========================================
# ENSEMBLE MODEL
# =========================================

ensemble = VotingClassifier(
    estimators=[
        ('cat', cat_model),
        ('xgb', xgb_model),
        ('lgb', lgb_model)
    ],
    voting='soft'
)

ensemble.fit(X, y)

# =========================================
# PREDICTION
# =========================================

test_preds = ensemble.predict(X_test)

submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Transported': test_preds.astype(bool)
})

submission.to_csv('submission.csv', index=False)

print("\nsubmission.csv generated successfully!")

[LightGBM] [Info] Number of positive: 3502, number of negative: 3452
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002708 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1902
[LightGBM] [Info] Number of data points in the train set: 6954, number of used features: 17
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503595 -> initscore=0.014380
[LightGBM] [Info] Start training from score 0.014380
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -i